In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
sns.set_style('whitegrid')
sns.set_palette("Set2")

%matplotlib inline

# Leer los datos

In [49]:
totales = pd.read_csv("../../../data/respuestas_fede.csv")
print(totales.shape)

#globales
marmol = totales.loc[totales["escuela"] == "Colegio Modelo Mármol"]
mantovani = totales.loc[totales["escuela"] == "Escuela Nueva Juan Mantovani"]

file_ext = '.png'
dpi_value = 200

(369, 22)


##  Son copia o referencia vs Qué puedo hacer para q no vea la foto 

In [50]:
# Boilerplate para generar grafico de amiga_wpp vs amiga_no_ver_foto
def amiga_wpp_vs_wiki(df, df_name, agrupando = False):
    amiga_wpp         = df["amiga_wpp"]
    amiga_no_ver_foto = df["amiga_no_ver_foto"]

    def analizar_misc_amiga_wpp(val):
      if val == "La foto ahora existe en WhatsApp y mi amiga la puede ver cuando mira nuestro chat, pero no la tiene en su celular":
          return "Con Misc."
      if val == "Le mando una copia de mi foto.":
          return "Sin Misc."
      if val == "No sé":
          return "No sé"
      if val == "Le doy permiso para ver la foto que tengo guardada en mi celular.":
          return "Con Misc."
    
    def analizar_misc_amiga_no_ver_foto(val):
      if val == "Tengo que borrar la foto en el chat.":
          return "Con Misc."
      if val == "No puedo hacer nada, mi amiga ahora también tiene la foto y no tengo manera de sacársela.":
          return "Sin Misc."
      if val == "Tengo que borrar la foto en la Galería de fotos de mi celular.":
          return "Con Misc."
      if val == "No sé":
          return "No sé"

    if(agrupando):
        filas =    [
                    "Sin Misc.",
                    "No sé",
                    "Con Misc."
                   ]
        columnas = [
                    "Con Misc.",
                    'No sé',
                    "Sin Misc."
                   ]
        amiga_wpp =         amiga_wpp.apply(analizar_misc_amiga_wpp)
        amiga_no_ver_foto = amiga_no_ver_foto.apply(analizar_misc_amiga_no_ver_foto)
        cross_tab_result = pd.crosstab(amiga_wpp, amiga_no_ver_foto)
        
    else:
        filas =    [
                    "Le doy permiso\n para ver la foto\n en mi celu",
                    "La foto existe\n en Wpp pero\n no en su celu",
                    "No sé",
                    "Le mando\n una copia",
                   ]
        columnas = [
                    "No puedo\n hacer nada",
                    "No sé",
                    "Tengo que borrar\n la foto en el chat",
                    "Tengo que borrar\n la foto en mi celu",
                   ]
        cross_tab_result = pd.crosstab(amiga_wpp, amiga_no_ver_foto)
        cross_tab_result = cross_tab_result.rename(
            index={  "La foto ahora existe en WhatsApp y mi amiga la puede ver cuando mira nuestro chat, pero no la tiene en su celular": "La foto existe\n en Wpp pero\n no en su celu",
                     "Le mando una copia de mi foto.":                                                                                    "Le mando\n una copia",
                     "Le doy permiso para ver la foto que tengo guardada en mi celular.":                                                 "Le doy permiso\n para ver la foto\n en mi celu",
                     },
            columns={"Tengo que borrar la foto en el chat.":                                                                              "Tengo que borrar\n la foto en el chat",
                     "No puedo hacer nada, mi amiga ahora también tiene la foto y no tengo manera de sacársela.":                         "No puedo\n hacer nada",
                     "Tengo que borrar la foto en la Galería de fotos de mi celular.":                                                    "Tengo que borrar\n la foto en mi celu"
                     })

    # # filas   = arriba hacia abajo, mayores misconceptions a correcta
    # # columnas= izq a derecha,      correcta a mayores misconceptions
    cross_tab_result = cross_tab_result.reindex(filas, columns=columnas)

    def agregar_totales(fila):
       if(agrupando):
           return fila[0]+fila[1]+fila[2]
       else:
           return fila[0]+fila[1]+fila[2]+fila[3]
    
    def dividir(fila): 
       if(agrupando):
           return fila.div(fila[3])
       else:
           return fila.div(fila[4])

    cross_tab_result['totales'] = cross_tab_result.apply(agregar_totales, axis=1)
    cross_tab_result = cross_tab_result.apply(dividir, axis=1)
    cross_tab_result = cross_tab_result.drop(['totales'], axis=1)

    sns.heatmap(cross_tab_result, cmap='YlGnBu', annot=True, fmt='.2f', linewidths=0.5)
    plt.title('Relación de Respuestas\nFotos Wpp vs Borrar fotos\nDatos {} - Agrupando {}'.format(df_name,agrupando))
    plt.xlabel('Misconceptions Funcionalidad Wikipedia')
    plt.ylabel('Link wikipedia')
    # plt.show()
    plt.savefig('wpp_vs_no_ver_foto_{}_agrupando_{}'.format(df_name,agrupando)+file_ext, bbox_inches='tight', dpi=dpi_value)
    plt.clf()

In [51]:
# llamo al generador de graficos con los tres dfs
amiga_wpp_vs_wiki(totales, "totales")
amiga_wpp_vs_wiki(marmol, "marmol")
amiga_wpp_vs_wiki(mantovani, "mantovani")

# # Agrupando misc
amiga_wpp_vs_wiki(totales, "totales", True)
amiga_wpp_vs_wiki(marmol, "marmol", True)
amiga_wpp_vs_wiki(mantovani, "mantovani", True)

<Figure size 640x480 with 0 Axes>